# 第 2 周第 2 天练习 —— 量子百科问答（Ollama + Wikipedia + Gradio）

## 练习目标（理念）

做一个「有依据」的量子物理小助手：

1. 用 `requests` + BeautifulSoup 抓取若干维基百科页面，拼成 `wiki_context`
2. 把上下文塞进 user prompt，让本地 `llama3.2` **基于这些内容**回答
3. 用 Gradio 做简单 UI，并 **流式（streaming）** 一边生成一边刷新 Markdown

这是 Week 2 里常见的模式：**检索/抓取 → 塞进 prompt → 流式聊天界面**。

## 和本课 Week 2 的关系

| 本课概念 | 本练习里你会看到 |
|----------|------------------|
| 网页抓取与清洗 | `HEADERS`、`BeautifulSoup`、去掉 script/style 等标签 |
| 把外部知识放进 prompt | `wiki_context` 拼进 user 消息 |
| 流式输出 | `stream=True`，循环累加 `delta.content` 再 `yield` |
| Gradio Interface | `gr.Interface` + examples 一键试问 |

## 怎么跑

1. 本机 `ollama serve` 且已有 `llama3.2`
2. 能访问 `en.wikipedia.org`（抓取单元格会联网）
3. 从上到下运行；最后一格会 `launch` Gradio，浏览器里提问即可


In [ ]:
# ========== 导入：抓取、解析 HTML、调本地模型、搭 Gradio 界面 ==========

# 导入 requests：HTTP GET 维基百科页面
import requests
# 从 bs4 导入 BeautifulSoup：把 HTML 解析成可清洗的文档树
from bs4 import BeautifulSoup
# 从 openai 导入 OpenAI：通过 OpenAI 兼容协议调用本机 Ollama
from openai import OpenAI
# 导入 gradio：快速搭一个问答 Web UI
import gradio as gr


In [ ]:
# ========== 连接 Ollama：OpenAI 兼容客户端指向本机 /v1 ==========

# 连接 Ollama——无需 API Key（api_key 仅占位；真正推理在本地）
ollama = OpenAI(
    api_key="ollama",
    base_url="http://localhost:11434/v1"
)

# 选用的本地模型名；须与 ollama list 里已有名称一致
MODEL = "llama3.2"


In [ ]:
# ========== 抓取维基百科：多页面拼成 wiki_context，作为后续问答的「依据」 ==========

# 浏览器 User-Agent：有些站点对默认 Python UA 不友好，伪装成常见 Chrome 更稳
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36"
}

# 主题相关的维基页面字典：key 便于日志，value 是完整 URL（勿改 URL，改了就换数据源）
WIKI_PAGES = {
    "quantum":           "https://en.wikipedia.org/wiki/Quantum",
    "quantum_mechanics":  "https://en.wikipedia.org/wiki/Quantum_mechanics",
    "quantum_computing":  "https://en.wikipedia.org/wiki/Quantum_computing",
    "quantum_entanglement": "https://en.wikipedia.org/wiki/Quantum_entanglement",
    "quantum_field_theory": "https://en.wikipedia.org/wiki/Quantum_field_theory",
    "planck_constant":    "https://en.wikipedia.org/wiki/Planck_constant",
    "wave_function":      "https://en.wikipedia.org/wiki/Wave_function",
}

def fetch_page(url, max_chars=1500):
    # 抓取单个 URL，清洗正文，并截断到 max_chars，避免 prompt 过长
    try:
        # timeout=10：避免某页卡住拖死整格
        resp = requests.get(url, headers=HEADERS, timeout=10)
        # 用 html.parser 解析响应字节内容
        soup = BeautifulSoup(resp.content, "html.parser")
        # 取 <title>；没有则用占位 "No title"（文案保持英文）
        title = soup.title.string.strip() if soup.title else "No title"
        if soup.body:
            # 删掉脚本/样式/导航等噪声节点，减少无关文本进入 prompt
            for tag in soup.body(["script", "style", "img", "input", "nav", "footer"]):
                tag.decompose()
            # 抽出纯文本；段落之间用换行分隔，并 strip 空白
            text = soup.body.get_text(separator="\n", strip=True)
        else:
            # 没有 body 就当空页
            text = ""
        # 返回带标题与 URL 头的片段，再整体截断
        return f"Page: {title}\nURL: {url}\n\n{text}"[:max_chars]
    except Exception as e:
        # 失败时返回可读错误串（英文模板保持原样），让上层仍能拼 context
        return f"Could not fetch {url}: {e}"

def fetch_wiki_context():
    # 遍历 WIKI_PAGES，逐页 fetch，再用分隔线拼成一大段上下文
    parts = []
    for name, url in WIKI_PAGES.items():
        # 打印进度，方便知道卡在哪一页
        print(f"Fetching {name} page...")
        parts.append(fetch_page(url))
    # 页与页之间用 Markdown 风格水平线分隔
    return "\n\n---\n\n".join(parts)

# 加载阶段：抓取全部页面，结果存进全局 wiki_context，供后面 stream_ollama 使用
print("Loading Wikipedia Quantum content...")
wiki_context = fetch_wiki_context()
# 打印总字符数；源码里的 \! 保持原样（历史转义写法）
print(f"Done\! Fetched {len(wiki_context)} characters of content.")


In [ ]:
# ========== system message：定「量子专家」人设与回答风格（发给模型的英文指令勿译）==========

system_message = """You are a quantum physics expert assistant.
You explain quantum concepts clearly, from beginner to advanced level.
Your answers are grounded in the Wikipedia content provided.
Be concise, accurate, and engaging. Respond in markdown without code blocks."""


In [ ]:
# ========== 流式问答：把问题 + wiki_context 塞进 messages，边生成边 yield 累积文本 ==========

def stream_ollama(question, model_name):
    # 组装 user prompt：先说明任务，再写 Question，再附 Wikipedia content（英文模板保持原样）
    user_prompt = "Based on the following Wikipedia content about Quantum, answer this question:\n\n"
    user_prompt += f"Question: {question}\n\n"
    user_prompt += f"Wikipedia content:\n{wiki_context}"

    # system 定角色；user 带问题与依据
    messages = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": user_prompt}
    ]

    # stream=True：服务端按增量推送；model_name 来自 Gradio 下拉框
    stream = ollama.chat.completions.create(
        model=model_name,
        messages=messages,
        stream=True
    )

    # result 累积已生成文本；每次 yield 完整前缀，Gradio Markdown 才能「打字机」刷新
    result = ""
    for chunk in stream:
        # delta.content 可能为 None（例如结束块），用 or "" 避免把 None 拼进去
        result += chunk.choices[0].delta.content or ""
        yield result


In [ ]:
# ========== Gradio Interface：输入问题 + 选模型，输出流式 Markdown 答案 ==========

# 多行文本框：用户输入量子相关问题（label/info 英文是 UI 文案，保持原样）
question_input = gr.Textbox(
    label="Your question about Quantum:",
    info="Ask anything about quantum physics, mechanics, computing, or entanglement",
    lines=3
)
# 下拉选择 Ollama 模型；目前列表只有 llama3.2，与上面 MODEL 一致
model_selector = gr.Dropdown(
    ["llama3.2"],
    label="Ollama model",
    value="llama3.2"
)
# 答案区用 Markdown 组件展示（流式时会反复更新）
answer_output = gr.Markdown(label="Answer:")

# 把 stream_ollama 接到 Interface：inputs 两路，outputs 一路
view = gr.Interface(
    fn=stream_ollama,
    title="Quantum Expert (powered by Ollama + Wikipedia)",
    description="Ask questions about quantum physics — answers grounded in Wikipedia content.",
    inputs=[question_input, model_selector],
    outputs=[answer_output],
    # examples：点击即可填入示例问题 + 模型名（字符串保持原样）
    examples=[
        ["what is quantum mechanics", "llama3.2"],
        ["what is quantum entanglement", "llama3.2"],
        ["what is quantum computing", "llama3.2"],
        ["what is the Planck constant", "llama3.2"],
        ["what is a wave function", "llama3.2"],
        ["what is quantum field theory", "llama3.2"],
    ],
    # 关闭 flagging（举报）按钮，界面更干净
    flagging_mode="never"
)
# inbrowser=True：启动后尝试自动打开浏览器
view.launch(inbrowser=True)
